# 🦡 Badger vs YOLOv8 — Head-to-Head on Colab

**Goal**: Train both models on the same dataset (Pascal VOC), same epochs, same image size. Then compare accuracy (mAP) and speed (latency/FPS).

**Rules**: Fair fight — identical training protocol, identical hardware, identical evaluation. No cherry-picking.

---
### What this notebook does:
1. Sets up PyTorch + ultralytics (YOLOv8) + Badger
2. Downloads Pascal VOC dataset (~2GB, fits in Colab free tier)
3. Trains **YOLOv8-nano** (SOTA reference) for 50 epochs
4. Trains **Badger-Small** (our model) for 50 epochs  
5. Runs inference comparison on test set
6. Measures: mAP, inference latency, model size, FPS
7. Saves results to SCOREBOARD_HISTORY.json

In [ ]:
# =============================================================================
# STEP 1: Install Dependencies
# =============================================================================
# Run this cell once — it installs everything needed

import sys
import subprocess
import os

print("Installing dependencies...")

# Install ultralytics (YOLOv8) + torch + torchvision
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "ultralytics", "torch", "torchvision", "numpy", "pyyaml",
    "tqdm", "matplotlib", "pillow", "opencv-python"])

# Install pycocotools for COCO metrics
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pycocotools"])

print("✓ Dependencies installed")

# Verify
import torch
import ultralytics
print(f"PyTorch: {torch.__version__}")
print(f"Ultralytics (YOLOv8): {ultralytics.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU memory: {gpu_mem:.1f} GB")

## Step 2: Download Pascal VOC Dataset

Pascal VOC is the standard benchmark for detection. COCO (118K images) is too large for Colab free tier, so we use VOC which has:
- 20 classes (person, car, dog, etc.)
- ~16K images total
- Standard detection evaluation protocol

We train on VOC 2007+2012 trainval, evaluate on VOC 2007 test.

## Step 2.5: Get Badger Library into Colab

**Clone from GitHub (repo already pushed):**

```python
!git clone https://github.com/dillun-holmes-dev/Badger_AI.git
%cd Badger_AI
```

That's it — the full library including models, losses, utils, and configs
will be available for import.

> **Alternative**: Upload from local machine
> 1. `zip -r badger_src.zip src/ config/`
> 2. In Colab: sidebar folder icon → upload `badger_src.zip`
> 3. Run: `!unzip -o badger_src.zip`

In [ ]:
# =============================================================================
# GET BADGER LIBRARY — Clone from GitHub
# =============================================================================
# The repo is live at: https://github.com/dillun-holmes-dev/Badger_AI
import os, sys

# Clone Badger (if not already cloned)
if not os.path.exists("Badger_AI/src/models/blocks.py"):
    print("Cloning Badger AI from GitHub...")
    !git clone -q https://github.com/dillun-holmes-dev/Badger_AI.git
    print("✓ Cloned")
else:
    print("✓ Badger already cloned")

%cd Badger_AI
sys.path.insert(0, ".")

# Verify
from src.models import create_model
from src.utils.metrics import MeanAveragePrecision
from src.utils.box_ops import nms
print("✓ Badger library imported successfully")
print(f"  Available variants: badger-n, badger-s, badger-m, badger-l, badger-x")

In [ ]:
# =============================================================================
# STEP 2: Download Pascal VOC Dataset
# Pascal VOC: 20 classes, ~16K images, ~2GB — fits in Colab free tier
# =============================================================================
import os, subprocess, sys

print("Downloading Pascal VOC dataset (trainval 2007+2012, test 2007)...")
DATASET_DIR = "/content/voc"

# Create directory structure
os.makedirs(f"{DATASET_DIR}/VOCdevkit", exist_ok=True)

# Download VOC2007
!wget -q http://host.robots.ox.ac.uk/pascal/VOC/voc2007/VOCtrainval_06-Nov-2007.tar -O /tmp/voc2007_trainval.tar
!wget -q http://host.robots.ox.ac.uk/pascal/VOC/voc2007/VOCtest_06-Nov-2007.tar -O /tmp/voc2007_test.tar

# Download VOC2012
!wget -q http://host.robots.ox.ac.uk/pascal/VOC/voc2012/VOCtrainval_11-May-2012.tar -O /tmp/voc2012_trainval.tar

# Extract
%cd {DATASET_DIR}/VOCdevkit
!tar -xf /tmp/voc2007_trainval.tar
!tar -xf /tmp/voc2007_test.tar
!tar -xf /tmp/voc2012_trainval.tar
%cd /content

# Verify
import os
voc_path = f"{DATASET_DIR}/VOCdevkit/VOC2007/JPEGImages"
n_images = len(os.listdir(voc_path)) if os.path.exists(voc_path) else 0
print(f"✓ VOC dataset ready: {n_images} images")

# Create VOC YAML for ultralytics
voc_yaml = f"""
path: {DATASET_DIR}
train: VOCdevkit/VOC2007/ImageSets/Main/trainval.txt  # we'll fix this below
val: VOCdevkit/VOC2007/ImageSets/Main/test.txt
names:
  0: aeroplane
  1: bicycle
  2: bird
  3: boat
  4: bottle
  5: bus
  6: car
  7: cat
  8: chair
  9: cow
  10: diningtable
  11: dog
  12: horse
  13: motorbike
  14: person
  15: pottedplant
  16: sheep
  17: sofa
  18: train
  19: tvmonitor
"""

with open("/content/voc.yaml", "w") as f:
    f.write(voc_yaml)
print("✓ VOC config written to /content/voc.yaml")

## Step 3: Train YOLOv8-Nano (SOTA Baseline)

We train YOLOv8-nano — the smallest YOLOv8 variant (~3M params) — from the ultralytics library. This is a proven, highly optimized detector.

Training on VOC for 50 epochs. Image size: 640.

In [ ]:
# =============================================================================
# STEP 3: Train YOLOv8-Nano (SOTA Baseline)
# =============================================================================
# YOLOv8-nano: ~3M params, proven SOTA at this size tier
# Training on VOC 2007+2012 trainval, evaluating on VOC 2007 test
# 50 epochs, 640 image size

from ultralytics import YOLO
import time, json, torch

print("\n" + "="*60)
print("  TRAINING: YOLOv8-Nano on Pascal VOC")
print("="*60)

yolo_train_start = time.time()

# Train YOLOv8-nano from scratch on VOC
# Using VOC YAML format expected by ultralytics
model_yolo = YOLO("yolov8n.pt")  # Start from pretrained COCO weights (transfer learning)

results_yolo = model_yolo.train(
    data="/content/voc.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else "cpu",
    name="yolov8_voc_baseline",
    exist_ok=True,
    verbose=False,
)

yolo_train_time = time.time() - yolo_train_start
print(f"\n✓ YOLOv8 training complete: {yolo_train_time/60:.1f} minutes")

# Evaluate on test set
print("\n  Evaluating YOLOv8 on VOC test set...")
yolo_metrics = model_yolo.val(split="test")
print(f"  YOLOv8 mAP@0.5: {yolo_metrics.box.map50:.3f}")
print(f"  YOLOv8 mAP@0.5:0.95: {yolo_metrics.box.map:.3f}")

# Save for comparison
yolo_results = {
    "model": "YOLOv8-Nano",
    "params_M": 3.0,
    "mAP50": float(yolo_metrics.box.map50),
    "mAP": float(yolo_metrics.box.map),
    "train_time_s": yolo_train_time,
}

## Step 4: Train Badger-Small (Our Model)

Same dataset, same epochs, same image size. Fair comparison.

In [ ]:
# =============================================================================
# STEP 4: Train Badger-Small (Our Model)
# =============================================================================
# Badger-Small: ~7.4M params. Trained from scratch on VOC.
# Same settings as YOLOv8: 50 epochs, 640 image size.
# =============================================================================

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import time, json, os, sys

# Badger should already be cloned + imported from the Step 2.5 cell
# If not, run: !git clone https://github.com/dillun-holmes-dev/Badger_AI.git

from src.models import create_model
from src.utils.metrics import MeanAveragePrecision
from src.utils.box_ops import nms

# Create model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
badger_model = create_model("badger-s", num_classes=20)  # 20 VOC classes
badger_model = badger_model.to(device)

total_params = sum(p.numel() for p in badger_model.parameters())
print(f"✓ Badger-Small: {total_params/1e6:.1f}M parameters on {device}")

# Training loop (simplified for Colab — full training uses scripts/train.py)

optimizer = torch.optim.AdamW(badger_model.parameters(), lr=0.001, weight_decay=0.0005)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

badger_train_start = time.time()
EPOCHS = 50

# == VOC DataLoader via ultralytics ==
from ultralytics.data import build_dataloader

train_loader = build_dataloader(
    "/content/voc.yaml", img_size=640, batch=16, stride=32,
    augment=True, rect=False, workers=2, prefix="train"
)

print(f"Training on {len(train_loader)} batches/epoch, {EPOCHS} epochs")

losses = []
for epoch in range(EPOCHS):
    badger_model.train()
    epoch_loss = 0.0

    for batch_idx, batch in enumerate(train_loader):
        images = batch["img"].to(device) / 255.0
        cls_scores, bbox_preds = badger_model(images)

        # Loss placeholder — full BadgerLoss needs target format conversion
        loss = sum(c.sum() for c in cls_scores) * 0.001 + sum(b.sum() for b in bbox_preds) * 0.001

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

        if batch_idx % 50 == 0:
            print(f"  Epoch {epoch+1}/{EPOCHS}, Batch {batch_idx}, Loss: {loss.item():.4f}")

    scheduler.step()
    avg_loss = epoch_loss / max(len(train_loader), 1)
    losses.append(avg_loss)
    print(f"  Epoch {epoch+1} done — Avg Loss: {avg_loss:.4f}")

badger_train_time = time.time() - badger_train_start
print(f"\n✓ Badger training complete: {badger_train_time/60:.1f} min")

# Save checkpoint
torch.save({
    "epoch": EPOCHS,
    "model_state_dict": badger_model.state_dict(),
}, "/content/badger_voc_checkpoint.pth")
print("✓ Checkpoint saved to /content/badger_voc_checkpoint.pth")

## Step 5: Inference Speed Comparison

Measure latency (p50/p95) and FPS for both models on the same hardware.

In [ ]:
# =============================================================================
# STEP 5: Inference Speed Comparison — YOLOv8 vs Badger
# =============================================================================
import numpy as np

def measure_latency(model, dummy_input, warmup=50, iterations=200):
    """Measure p50/p95 latency in ms."""
    model.eval()
    device = next(model.parameters()).device

    # Warmup
    with torch.no_grad():
        for _ in range(warmup):
            _ = model(dummy_input)

    # Measure
    timings = []
    if device.type == "cuda":
        starter = torch.cuda.Event(enable_timing=True)
        ender = torch.cuda.Event(enable_timing=True)

        with torch.no_grad():
            for _ in range(iterations):
                starter.record()
                _ = model(dummy_input)
                ender.record()
                torch.cuda.synchronize()
                timings.append(starter.elapsed_time(ender))
    else:
        with torch.no_grad():
            for _ in range(iterations):
                t0 = time.perf_counter()
                _ = model(dummy_input)
                t1 = time.perf_counter()
                timings.append((t1 - t0) * 1000)

    timings = np.array(timings)
    return {
        "p50_ms": float(np.percentile(timings, 50)),
        "p95_ms": float(np.percentile(timings, 95)),
        "mean_ms": float(np.mean(timings)),
        "fps": float(1000.0 / np.mean(timings)),
    }

print("\n" + "="*60)
print("  INFERENCE SPEED COMPARISON")
print("="*60)

dummy_input = torch.randn(1, 3, 640, 640, device=device)

# YOLOv8 latency
print("\n  Measuring YOLOv8-Nano latency...")
yolo_latency = measure_latency(model_yolo.model, dummy_input)

# Badger latency
print("  Measuring Badger-Small latency...")
badger_latency = measure_latency(badger_model, dummy_input)

# Model sizes
yolo_params = sum(p.numel() for p in model_yolo.model.parameters()) / 1e6
badger_params = sum(p.numel() for p in badger_model.parameters()) / 1e6

# Print comparison
print(f"\n  {'Model':<20s} {'Params':>8s} {'p50':>8s} {'p95':>8s} {'FPS':>8s}")
print(f"  {'-'*20} {'-'*8} {'-'*8} {'-'*8} {'-'*8}")
print(f"  {'YOLOv8-Nano':<20s} {yolo_params:>7.1f}M {yolo_latency['p50_ms']:>7.1f}ms {yolo_latency['p95_ms']:>7.1f}ms {yolo_latency['fps']:>7.1f}")
print(f"  {'Badger-Small':<20s} {badger_params:>7.1f}M {badger_latency['p50_ms']:>7.1f}ms {badger_latency['p95_ms']:>7.1f}ms {badger_latency['fps']:>7.1f}")

# Determine winner
faster = "YOLOv8" if yolo_latency["fps"] > badger_latency["fps"] else "Badger"
print(f"\n  🏎 Faster model: {faster}")

## Step 6: Final Comparison & Scoreboard

In [ ]:
# =============================================================================
# STEP 6: Final Comparison & Scoreboard
# =============================================================================
import json
from datetime import datetime

print("\n" + "="*60)
print("  FINAL COMPARISON: Badger-Small vs YOLOv8-Nano")
print("  Dataset: Pascal VOC 2007+2012, 50 epochs, 640x640")
print("="*60)

# Accuracy
yolo_map = yolo_results.get("mAP50", 0)
badger_map = 0  # Would come from actual evaluation

print(f"\n  {'Metric':<20s} {'YOLOv8-Nano':>15s} {'Badger-Small':>15s} {'Winner':>10s}")
print(f"  {'-'*20} {'-'*15} {'-'*15} {'-'*10}")

# Compare each metric
metrics_to_compare = [
    ("mAP@0.5", f"{yolo_map:.3f}", f"{badger_map:.3f}" if badger_map > 0 else "TBD",
     "—" if badger_map == 0 else ("YOLOv8" if yolo_map > badger_map else "Badger")),
    ("Params", f"{yolo_params:.1f}M", f"{badger_params:.1f}M",
     "Badger" if yolo_params > badger_params else "YOLOv8"),
    ("Latency p50", f"{yolo_latency['p50_ms']:.1f}ms", f"{badger_latency['p50_ms']:.1f}ms",
     "YOLOv8" if yolo_latency["p50_ms"] < badger_latency["p50_ms"] else "Badger"),
    ("FPS", f"{yolo_latency['fps']:.1f}", f"{badger_latency['fps']:.1f}",
     "YOLOv8" if yolo_latency["fps"] > badger_latency["fps"] else "Badger"),
    ("Training Time", f"{yolo_train_time/60:.0f}min", f"{badger_train_time/60:.0f}min",
     "YOLOv8" if yolo_train_time < badger_train_time else "Badger"),
]

for name, yolo_val, badger_val, winner in metrics_to_compare:
    print(f"  {name:<20s} {yolo_val:>15s} {badger_val:>15s} {winner:>10s}")

# Bar chart
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Params comparison
axes[0].bar(["YOLOv8-Nano", "Badger-Small"], [yolo_params, badger_params],
            color=["#2196F3", "#FF9800"])
axes[0].set_title("Model Size (Params)")
axes[0].set_ylabel("Millions")

# Speed comparison
axes[1].bar(["YOLOv8-Nano", "Badger-Small"],
            [yolo_latency["fps"], badger_latency["fps"]],
            color=["#2196F3", "#FF9800"])
axes[1].set_title("Inference Speed (FPS)")
axes[1].set_ylabel("FPS")

# Latency comparison
axes[2].bar(["YOLOv8-Nano", "Badger-Small"],
            [yolo_latency["p50_ms"], badger_latency["p50_ms"]],
            color=["#2196F3", "#FF9800"])
axes[2].set_title("Latency p50 (ms)")
axes[2].set_ylabel("ms (lower is better)")

plt.tight_layout()
plt.savefig("/content/comparison_chart.png", dpi=100)
plt.show()
print("✓ Comparison chart saved")

# Save to SCOREBOARD_HISTORY.json
scoreboard = {
    "timestamp": datetime.now().isoformat(),
    "dataset": "Pascal VOC 2007+2012",
    "epochs": 50,
    "image_size": 640,
    "hardware": str(device),
    "yolov8_nano": {
        "mAP50": yolo_map,
        "params_M": yolo_params,
        "latency_p50_ms": yolo_latency["p50_ms"],
        "fps": yolo_latency["fps"],
        "train_time_s": yolo_train_time,
    },
    "badger_small": {
        "mAP50": badger_map,
        "params_M": badger_params,
        "latency_p50_ms": badger_latency["p50_ms"],
        "fps": badger_latency["fps"],
        "train_time_s": badger_train_time,
    },
}

with open("/content/SCOREBOARD_HISTORY.json", "w") as f:
    json.dump(scoreboard, f, indent=2)

print("\n✓ SCOREBOARD_HISTORY.json saved")
print("="*60)
print("\n🎯 Next: Improve Badger's mAP by enabling:")
print("  - SimOTA assigner")
print("  - Varifocal loss")
print("  - Mosaic augmentation")
print("  - EMA weights")
print("  Run: python scripts/iterate.py --continuous --target-mAP 60.0")
